# Import Libraries

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import plotly.io as pio
import plotly.offline as pyo
import plotly.subplots as sp

import os
import warnings
warnings.filterwarnings('ignore')
import time
import random


import sklearn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score, GridSearchCV

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# Exploratory Data Analysis

In [10]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')


In [11]:
train.head()
train.info()
train.describe()

<class 'pandas.DataFrame'>
RangeIndex: 668665 entries, 0 to 668664
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           668665 non-null  int64  
 1   Age                          668665 non-null  int64  
 2   Annual_Income_USD            668665 non-null  float64
 3   Daily_Commute_km             668665 non-null  float64
 4   Number_of_Cars_Owned         668665 non-null  int64  
 5   Charging_Stations_Near_Home  668665 non-null  int64  
 6   Charging_Stations_Near_Work  668665 non-null  int64  
 7   Environmental_Concern_Level  668665 non-null  float64
 8   Gender                       668665 non-null  str    
 9   City_Type                    668665 non-null  str    
 10  Current_Car_Type             668665 non-null  str    
 11  Home_Charging_Possible       668665 non-null  str    
 12  Subsidy_Available            668665 non-null  str    
 13  Range_Anxi

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level
count,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000
mean,334332.000000,47.039171,84769.266989,32.158298,1.712626,4.960408,7.176314,2.935477
std,193027.103211,12.875448,28648.029042,18.730474,0.729275,3.926843,5.186627,1.429119
min,0.000000,25.000000,30000.000000,5.000000,1.000000,0.000000,0.000000,1.000000
25%,167166.000000,36.000000,67376.000000,17.200000,1.000000,2.000000,3.000000,2.000000
50%,334332.000000,47.000000,84880.000000,33.600000,2.000000,4.000000,6.000000,3.000000
75%,501498.000000,58.000000,102753.000000,47.400000,2.000000,7.000000,10.000000,4.000000
max,668664.000000,69.000000,188549.000000,98.700000,4.000000,14.000000,19.000000,5.000000


In [12]:
train.isnull().sum()

id                             0
Age                            0
Annual_Income_USD              0
Daily_Commute_km               0
Number_of_Cars_Owned           0
Charging_Stations_Near_Home    0
Charging_Stations_Near_Work    0
Environmental_Concern_Level    0
Gender                         0
City_Type                      0
Current_Car_Type               0
Home_Charging_Possible         0
Subsidy_Available              0
Range_Anxiety_Level            0
Will_Buy_EV                    0
dtype: int64

In [ ]:
#Preparing features and target variable for modeling
target = "Will_Buy_EV" 

# roc_auc_score requires the target variable to be binary, so we will convert "Yes" to 1 and "No" to 0
y = (train[target] == "Yes").astype(int)
X = train.drop(columns=["id", target]) #dropping the id and target variable from the features
X_test = test.drop(columns=["id"])

x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


# LightGBM

In [16]:
# LightGBM can handle categorical columns natively if they're cast to the
# pandas "category" dtype, avoiding manual one-hot/label encoding.
cat_cols = X.select_dtypes(include="str").columns.tolist()
for c in cat_cols:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")

In [19]:
# LightGBM hyperparameters. Nothing exotic here: a fairly standard set of
# regularization knobs (feature/bagging fraction) to reduce overfitting,
# plus a moderate learning rate paired with early stopping.
params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.06,
    "num_leaves": 63,
    "min_child_samples": 30,
    "feature_fraction": 0.8,   # use 80% of features per tree
    "bagging_fraction": 0.8,   # use 80% of rows per iteration
    "bagging_freq": 1,
    "verbosity": -1,
    "seed": 42,
}

# Cross-validation Setup

In [18]:
# Stratified K-fold preserves the ~17.5% positive class rate in every fold,
# which matters for a moderately imbalanced target like this one.
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
 
# oof (out-of-fold) predictions let us score the model on data it never
# trained on, without needing a separate holdout set.
oof = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []

In [ ]:
from sklearn.metrics import roc_auc_score #import roc_auc_score for evaluation

# Train one model per fold
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
 
    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
    dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols)
 
    model = lgb.train(
        params,
        dtrain,
        num_boost_round=500,
        valid_sets=[dval],
        # Stop training if validation AUC hasn't improved in 40 rounds,
        # to avoid overfitting and save runtime.
        callbacks=[lgb.early_stopping(40, verbose=False)],
    )
    
 
    # Record this fold's validation predictions into the oof array so every
    # row ends up with exactly one out-of-fold prediction.
    oof[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    fold_auc = roc_auc_score(y_val, oof[val_idx])
    fold_scores.append(fold_auc)
    print(f"Fold {fold}: AUC = {fold_auc:.5f} (best_iter={model.best_iteration})", flush=True)
 
    # Average this fold's test-set predictions across all folds
    # (a simple form of ensembling that tends to reduce variance).
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / n_splits

Fold 0: AUC = 0.94036 (best_iter=329)
Fold 1: AUC = 0.94130 (best_iter=173)
Fold 2: AUC = 0.94276 (best_iter=310)
Fold 3: AUC = 0.94221 (best_iter=232)
Fold 4: AUC = 0.94167 (best_iter=311)


# Report Cross-Validation Performance

In [23]:
overall_auc = roc_auc_score(y, oof)
print(f"\nOOF AUC: {overall_auc:.5f}")
print(f"Fold mean: {np.mean(fold_scores):.5f} +/- {np.std(fold_scores):.5f}")


OOF AUC: 0.94165
Fold mean: 0.94166 +/- 0.00082


# Submission

In [27]:
submission = pd.DataFrame({"id": test["id"], "Will_Buy_EV": test_preds})
submission.to_csv("/Users/khuynh436gmail.com/Desktop/Predicting-Electric-Vehicle-Purchase/submission.csv", index=False)
print("\nSaved submission.csv")


Saved submission.csv
